In [1]:
import pandas as pd
import torch
from torch_geometric.transforms import RandomLinkSplit
from sklearn.manifold import TSNE
import plotly.express as px
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

/home/giobbi/DDI_LLM_repo/DDI_with_ML/.venv/lib/python3.10/site-packages/torch_geometric/typing.py:68: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /home/giobbi/DDI_LLM_repo/DDI_with_ML/.venv/lib/python3.10/site-packages/libpyg.so: undefined symbol: _ZNK5torch8autograd4Node4nameEv
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/home/giobbi/DDI_LLM_repo/DDI_with_ML/.venv/lib/python3.10/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /home/giobbi/DDI_LLM_repo/DDI_with_ML/.venv/lib/python3.10/site-packages/torch_scatter/_version_cpu.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/home/giobbi/DDI_LLM_repo/DDI_with_ML/.venv/lib/python3.10/site-packages/torch_geometric/typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its us

In [2]:
from ddi_graph_neural_network.train_model import main

In [3]:
def get_node_embeddings(model, data):
    model.eval()
    with torch.no_grad():
        node_embeddings = model.encode(data.x, data.edge_index)
    node_embeddings = node_embeddings.cpu().numpy()
    return node_embeddings

def get_reduced_embeddings(node_embeddings):
    # Initialize t-SNE with 2 components for 2D visualization
    tsne = TSNE(n_components=2 ) #, random_state=42) # 42

    # Apply t-SNE to the drug embeddings
    reduced_embeddings = tsne.fit_transform(node_embeddings)

    return pd.DataFrame(reduced_embeddings, columns=['TSNE-1', 'TSNE-2'])

def plot_embeddings(reduced_embeddings_df):
    # Create a scatter plot using Plotly
    fig = px.scatter(
        reduced_embeddings_df,
        x='TSNE-1',
        y='TSNE-2',
        title='t-SNE Visualization of Drug Embeddings',
        width=800,
        height=600
    )
    fig.show()
    #return fig

In [4]:
#globals().update(main()) # Pass local variables from main() to the global scope

In [5]:
vars = main()

======== GPT+Desc | LR: 0.0003 ========
-------------------------------
Training with LR: 0.0003
Epoch: 001, Loss: 0.6876, Val: 0.8597
Epoch: 002, Loss: 0.6838, Val: 0.8624
Epoch: 003, Loss: 0.6797, Val: 0.8620
Epoch: 004, Loss: 0.6765, Val: 0.8634
Epoch: 005, Loss: 0.6762, Val: 0.8637
Epoch: 006, Loss: 0.6775, Val: 0.8636
Epoch: 007, Loss: 0.6774, Val: 0.8640
Epoch: 008, Loss: 0.6758, Val: 0.8648
Epoch: 009, Loss: 0.6749, Val: 0.8645
Epoch: 010, Loss: 0.6748, Val: 0.8668
Epoch: 011, Loss: 0.6752, Val: 0.8661
Epoch: 012, Loss: 0.6748, Val: 0.8672
Epoch: 013, Loss: 0.6753, Val: 0.8671
Epoch: 014, Loss: 0.6745, Val: 0.8684
Epoch: 015, Loss: 0.6742, Val: 0.8680
Epoch: 016, Loss: 0.6735, Val: 0.8692
Epoch: 017, Loss: 0.6731, Val: 0.8683
Epoch: 018, Loss: 0.6729, Val: 0.8692
Epoch: 019, Loss: 0.6728, Val: 0.8702
Epoch: 020, Loss: 0.6729, Val: 0.8700
Epoch: 021, Loss: 0.6727, Val: 0.8699
Epoch: 022, Loss: 0.6725, Val: 0.8716
Epoch: 023, Loss: 0.6720, Val: 0.8696
Epoch: 024, Loss: 0.6720, Val

In [7]:
graph_with_emb = vars['graph_data']
model = vars['model']
node_id_map = vars['node_id_map']

In [8]:
reversed_node_id_map = {v: k for k, v in node_id_map.items()}
lambda x : reversed_node_id_map(x)

<function __main__.<lambda>(x)>

In [12]:
graph_with_emb.x.shape

torch.Size([1491, 1536])

In [13]:
graph_with_emb.edge_index.shape

torch.Size([2, 95028])

In [38]:
emb = pd.read_csv("/data/giobbi/embeddings/Dr_Desc_GPT.csv", sep="\t", index_col=0).dropna()

In [39]:
graph_with_emb.edge_index.T

tensor([[ 631,  707],
        [ 412,  590],
        [ 923, 1302],
        ...,
        [ 831,  447],
        [ 983,  591],
        [1180,  488]])

In [40]:
idx = 707
drug_id =reversed_node_id_map[idx]
drug_id

'DB00966'

In [41]:
graph_with_emb.x[idx, :]

tensor([-0.0291, -0.0075, -0.0060,  ..., -0.0138, -0.0214, -0.0244])

In [42]:
emb[emb['Drug ID'] == drug_id]

,Drug ID,Drug Name,Discription,0,1,2,3,4,5,6,...,1526,1527,1528,1529,1530,1531,1532,1533,1534,1535
947,DB00966,Telmisartan,Telmisartan is an angiotensin II receptor anta...,-0.029059,-0.007466,-0.006039,-0.035208,-0.010313,0.035856,-0.021522,...,0.003267,0.010644,0.047973,-0.020834,-0.04722,0.015567,0.000414,-0.013764,-0.021366,-0.024376


In [43]:
embedding = get_node_embeddings(model, graph_with_emb)
embedding = get_reduced_embeddings(embedding)

In [44]:
# Map dataframe index (node integer ids) to drug IDs using reversed_node_id_map.
# Use dict.get to avoid KeyError for any unmapped indices and fall back to a placeholder.
embedding.index = embedding.index.map(lambda x: reversed_node_id_map.get(int(x), f"unknown_{int(x)}"))

In [45]:
plot_embeddings(embedding)

In [30]:
# TODO: add labels to the plot using drugID_DESC -> Make sure the order of embeddings matches the order of drugID_DESC
# TODO: color by drug category if available

In [30]:
# Create a dummy DataFrame
df = pd.DataFrame({
    'drug_id': ['DB00001', 'DB00002', 'DB00003'],
    'value': [10, 20, 30]
})

print("Original DataFrame:")
print(df)

#df = df[df['drug_id'] != 'DB00002']

# Set 'drug_id' as the index
df_indexed = df.set_index('drug_id')
print("\nAfter set_index('drug_id'):")
print(df_indexed)

# Reset the index back to default integer index
df_reset = df_indexed.reset_index()
print("\nAfter reset_index():")
print(df_reset)

df_reindexed = df_reset.reindex([0, 2, 1])
print("\nAfter reindexing to [0, 2]:")
print(df_reindexed)

Original DataFrame:
   drug_id  value
0  DB00001     10
1  DB00002     20
2  DB00003     30

After set_index('drug_id'):
         value
drug_id       
DB00001     10
DB00002     20
DB00003     30

After reset_index():
   drug_id  value
0  DB00001     10
1  DB00002     20
2  DB00003     30

After reindexing to [0, 2]:
   drug_id  value
0  DB00001     10
2  DB00003     30
1  DB00002     20
